In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import wandb

from data_center_model import DataCenterModel
from am_flexdc_training_utilities import train_model, evaluate_model

SEED = 0
EPOCHS = 150
LEARNING_RATE = 1e-4
BATCH_SIZE = 512
DATA_FILE = "../data/flexdc_all_data.csv"
MODEL_FILE = "../models/am_flexdc_condor_flexdc_objective_v1_state_dict.pt"
WANDB_ENTITY = "amenon06-boston-university"
WANDB_PROJECT = "flexdc-condor"
WANDB_RUN_NAME = "flexdc-objective-v1"
Path(MODEL_FILE).parent.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Makes the unchanged CONDOR model architecture.
#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
sim_model = DataCenterModel()
sim_model.to(device)
sim_model.train()

print("Device:", device)


In [ ]:
run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "architecture": "CONDOR DataCenterModel",
        "dataset": "FlexDC traditional-ISO pilot",
        "objective": "M_RSR + C_track + C_Qos",
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "train_fraction": 0.70,
        "heldout_fraction": 0.30,
        "device": str(device),
    },
)


In [ ]:
# Trains the three FlexDC objective components:
# [M_RSR, C_track, C_Qos].
# cross_validate=True keeps the paper-style deterministic 70/30 split.
sim_model, train_loss_record, heldout_loss_record = train_model(
    sim_model,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    verbose=True,
    cross_validate=True,
    data_file_path=DATA_FILE,
    wandb_run=run,
)


In [ ]:
# The logged FlexDC objective is reconstructed after prediction:
# C_total = M_RSR + C_track + C_Qos.


In [ ]:
plt.figure(figsize=(8, 4))
plt.title("FlexDC component MSE during training")
plt.plot(train_loss_record, label="Train")
plt.plot(heldout_loss_record, label="Held-out")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.legend()
plt.show()


In [ ]:
torch.save(sim_model.state_dict(), MODEL_FILE)
print(f"Saved model state dict to: {MODEL_FILE}")


In [ ]:
loaded_model = DataCenterModel()
loaded_model.load_state_dict(torch.load(MODEL_FILE, map_location=device))
loaded_model.to(device)
loaded_model.eval()

# Uses the same deterministic 70/30 split as train_model().
y_true_train, y_pred_train, y_true_test, y_pred_test = evaluate_model(
    loaded_model,
    cross_validate=True,
    data_file_path=DATA_FILE,
)


In [ ]:
plt.figure(figsize=(8, 4))
plt.title("Held-out Monetary RSR Cost (M_RSR)")
ind = np.argsort(y_true_test[:, 0])
plt.scatter(range(len(ind)), y_pred_test[ind, 0], alpha=0.02, label="Predicted")
plt.plot(range(len(ind)), y_true_test[ind, 0], alpha=0.7, label="Actual")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
plt.title("Held-out Ctrack Penalty")
ind = np.argsort(y_true_test[:, 1])
plt.scatter(range(len(ind)), y_pred_test[ind, 1], alpha=0.02, label="Predicted")
plt.plot(range(len(ind)), y_true_test[ind, 1], alpha=0.7, label="Actual")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
plt.title("Held-out QoS Penalty")
ind = np.argsort(y_true_test[:, 2])
plt.scatter(range(len(ind)), y_pred_test[ind, 2], alpha=0.02, label="Predicted")
plt.plot(range(len(ind)), y_true_test[ind, 2], alpha=0.7, label="Actual")
plt.legend()
plt.show()


In [ ]:
actual_c_total = y_true_test.sum(axis=1)
predicted_c_total = y_pred_test.sum(axis=1)

plt.figure(figsize=(8, 4))
plt.title("Held-out Full FlexDC Objective")
ind = np.argsort(actual_c_total)
plt.scatter(range(len(ind)), predicted_c_total[ind], alpha=0.02, label="Predicted")
plt.plot(range(len(ind)), actual_c_total[ind], alpha=0.7, label="Actual")
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

names = ["M_RSR", "C_track", "C_Qos"]
actual_total = y_true_test.sum(axis=1)
predicted_total = y_pred_test.sum(axis=1)
final_metrics = {}

for i, name in enumerate(names):
    actual = y_true_test[:, i]
    predicted = y_pred_test[:, i]
    mae = mean_absolute_error(actual, predicted)
    rmse = mean_squared_error(actual, predicted) ** 0.5
    r2 = r2_score(actual, predicted)

    print(name)
    print("  MAE: ", mae)
    print("  RMSE:", rmse)
    print("  R²:  ", r2)

    final_metrics[f"heldout_{name}_mae"] = mae
    final_metrics[f"heldout_{name}_rmse"] = rmse
    final_metrics[f"heldout_{name}_r2"] = r2

c_total_mae = mean_absolute_error(actual_total, predicted_total)
c_total_rmse = mean_squared_error(actual_total, predicted_total) ** 0.5
c_total_r2 = r2_score(actual_total, predicted_total)

print("C_total")
print("  MAE: ", c_total_mae)
print("  RMSE:", c_total_rmse)
print("  R²:  ", c_total_r2)

final_metrics["heldout_C_total_mae"] = c_total_mae
final_metrics["heldout_C_total_rmse"] = c_total_rmse
final_metrics["heldout_C_total_r2"] = c_total_r2
run.log(final_metrics)


In [ ]:
run.finish()
